## Human In the Loop

In [15]:
from dotenv import load_dotenv
import os

from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, END, START
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq
from langgraph.types import interrupt, Command

from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper

load_dotenv()


True

In [16]:
wiki_wrapper = WikipediaAPIWrapper(top_k_results=2 ,doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=wiki_wrapper, description="Find Information from Wikipedia Pages")

In [17]:
duck_duck_go = DuckDuckGoSearchRun(
    name="web_search_on_Duck_Duck_Go",
    description="Search the web for current events, recent news, live scores, and real-time information."
    )

In [18]:
@tool
def human_assiatance(query: str) -> str:
    """Ask the human user a question and wait for their answer.
    Use this whenever you need personal details or clarification from the user
    (e.g. their qualifications, experience, preferences). Do not ask such
    questions in a normal reply; use this tool instead."""
    human_response = interrupt({"query": query})
    return human_response["data"]

In [19]:
tools = [wiki, duck_duck_go, human_assiatance]

In [20]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [21]:
short_term_memory = MemorySaver()

In [22]:
model = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.2)

lllm_with_tools = model.bind_tools(tools)

In [23]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [24]:
def tool_calling_llm(state: State) -> State:
    return {'messages': [lllm_with_tools.invoke(state['messages'])  ]}

In [25]:
graph = StateGraph(State)

graph.add_node('tool_calling_llm', tool_calling_llm)
graph.add_node('tools', ToolNode(tools))

graph.set_entry_point('tool_calling_llm')
graph.add_conditional_edges('tool_calling_llm', tools_condition)
graph.add_edge('tools', 'tool_calling_llm')

build = graph.compile(checkpointer=short_term_memory)

In [27]:
config = {"configurable": {"thread_id": 7}}
messages = build.invoke({'messages': HumanMessage("Help me with my career by asking about my qualifications.")}, config=config)

for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Help me with my career by asking about my qualifications.
================================== Ai Message ==================================

I'd be happy to help with your career! Let me start by learning about your background.
Tool Calls:
  human_assiatance (1xznx5z6x)
 Call ID: 1xznx5z6x
  Args:
    query: To help guide your career, could you tell me about your qualifications? Specifically:

1. What is your highest level of education (degree, field of study, institution)?
2. How many years of professional experience do you have, and in what industry/role?
3. What are your key skills or areas of expertise?
4. Are you currently employed, looking for a new role, or considering a career change?
5. What are your career goals or what are you hoping to achieve?


In [28]:
config = {"configurable": {"thread_id": 7}}

human_response = "I am Computer science gratuate. having 2 years of exp, Software Engineering, No i resinged from my job, I want to build my own tech unicorn"
human_command = Command(resume={"data": human_response})

messages = build.invoke(human_command, config=config)

for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Help me with my career by asking about my qualifications.
================================== Ai Message ==================================

I'd be happy to help with your career! Let me start by learning about your background.
Tool Calls:
  human_assiatance (1xznx5z6x)
 Call ID: 1xznx5z6x
  Args:
    query: To help guide your career, could you tell me about your qualifications? Specifically:

1. What is your highest level of education (degree, field of study, institution)?
2. How many years of professional experience do you have, and in what industry/role?
3. What are your key skills or areas of expertise?
4. Are you currently employed, looking for a new role, or considering a career change?
5. What are your career goals or what are you hoping to achieve?
================================= Tool Message =================================
Name: human_assiatance

I am Computer science gratuate. having 2 years 